In [13]:
from plots_utils.loading import ExperimentConfig, load_experiment_results
from plots_utils.plot_availability_comparison import plot_availability_comparison
from plots_utils.plot_biased_unbiased_comparison import plot_biased_unbiased_comparison
from plots_utils.plot_av_mat import plot_av_mat
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

from matplotlib import pyplot as plt
# from plots_utils.loading import parse_tf_events_file
import numpy as np
import pandas as pd

from pathlib import Path
import os

def parse_tf_events_file(events_path, tag, time_horizon=None):
    """
    Returns the data in the file located in the folder events_paths,
    and corresponding to the tag.
    The tag can be: 'Train/Loss', 'Train/Metric', 'Test/Loss', 'Test/Metric'.
    """
    ea = EventAccumulator(events_path).Reload()
    # print(list(ea.Scalars(tag)))
    tag_values, steps = [], []
    for event in ea.Scalars(tag):
        if time_horizon is None or event.step <= time_horizon:
            tag_values.append(event.value)
            steps.append(event.step)
    return steps, tag_values

def get_exp_stats(config):
    results = list()
    for lr in config.lr_list:
        # time_horizon = time_horizons[p]  
        for algorithm in config.algorithms:
            # b_loop = config.b_values if algorithm == 'mixture' else [None] # in case we vary beta
            # for b in b_loop:
            for event in config.events:
                for seed in config.seeds:
                    for a in config.alphas:
                        for n_c in config.n_clients_list:
                            for av in config.availabilities:
                                for part in config.participations:
                                    for biased in config.biased_list:

                                        event_dir = config.get_event_dir(algorithm, lr, seed, 
                                                                            event, a, n_c, av, 
                                                                            config.n_rounds, part, biased, config.train_test)
                                        # print(event_dir)


                                        if os.path.exists(event_dir):
                                            # print('x')
                                            # _, values = parse_tf_events_file(event_dir, tag="Test/Metric", time_horizon=time_horizon)
                                            _, test_accuracy_values = parse_tf_events_file(event_dir, tag="Test/Metric")
                                            _, test_loss_values = parse_tf_events_file(event_dir, tag="Test/Loss")
                                            _, train_accuracy_values = parse_tf_events_file(event_dir, tag="Train/Metric")
                                            _, train_loss_values = parse_tf_events_file(event_dir, tag="Train/Loss")
                                            ### tag can be: 'Train/Loss', 'Train/Metric', 'Test/Loss', 'Test/Metric'
                                            max_accuracy = np.array(test_accuracy_values).max() * 100
                                            results.append({
                                                "algorithm": algorithm, 
                                                "availability": av,
                                                "alpha": a, 
                                                "participation": part,
                                                "max_test_accuracy": float(max_accuracy),
                                                "final_test_accuracy":test_accuracy_values[-1]*100,
                                                "test_accuracy": np.array(test_accuracy_values),
                                                "seed": seed,
                                                "lr": lr, "event": event, "n_clients": n_c,
                                                "biased": biased
                                            })

                                            # "b": float(b) if b else np.nan # in case we vary beta
    return pd.DataFrame(results)


main_folder = 'mnist_idle_accuracy_check'

availabilities="""
alphaF-50sl-1cb-1ft
alphaF-60sl-1cb-1ft
alphaF-70sl-1cb-1ft
alphaF-80sl-1cb-1ft
alphaF-90sl-1cb-1ft
alphaF-110sl-1cb-1ft
alphaF-130sl-1cb-1ft
alphaF-150sl-1cb-1ft
alphaF-170sl-1cb-1ft
alphaF-190sl-1cb-1ft
alphaF-40sl-2cb-1ft
alphaF-50sl-2cb-1ft
alphaF-60sl-2cb-1ft
alphaF-70sl-2cb-1ft
alphaF-80sl-2cb-1ft
alphaF-100sl-2cb-1ft
alphaF-120sl-2cb-1ft
alphaF-140sl-2cb-1ft
alphaF-160sl-2cb-1ft
alphaF-180sl-2cb-1ft
alphaF-30sl-3cb-1ft
alphaF-40sl-3cb-1ft
alphaF-50sl-3cb-1ft
alphaF-60sl-3cb-1ft
alphaF-70sl-3cb-1ft
alphaF-90sl-3cb-1ft
alphaF-110sl-3cb-1ft
alphaF-130sl-3cb-1ft
alphaF-150sl-3cb-1ft
alphaF-20sl-4cb-1ft
alphaF-30sl-4cb-1ft
alphaF-40sl-4cb-1ft
alphaF-50sl-4cb-1ft
alphaF-60sl-4cb-1ft
alphaF-80sl-4cb-1ft
alphaF-15sl-5cb-1ft
alphaF-25sl-5cb-1ft
alphaF-35sl-5cb-1ft
alphaF-45sl-5cb-1ft
alphaF-14sl-6cb-1ft
alphaF-24sl-6cb-1ft
alphaF-34sl-6cb-1ft
alphaF-13sl-7cb-1ft
alphaF-23sl-7cb-1ft
alphaF-50sl-1cb-3ft
alphaF-60sl-1cb-3ft
alphaF-70sl-1cb-3ft
alphaF-80sl-1cb-3ft
alphaF-90sl-1cb-3ft
alphaF-110sl-1cb-3ft
alphaF-130sl-1cb-3ft
alphaF-150sl-1cb-3ft
alphaF-170sl-1cb-3ft
alphaF-190sl-1cb-3ft
alphaF-40sl-2cb-3ft
alphaF-50sl-2cb-3ft
alphaF-60sl-2cb-3ft
alphaF-70sl-2cb-3ft
alphaF-80sl-2cb-3ft
alphaF-100sl-2cb-3ft
alphaF-120sl-2cb-3ft
alphaF-140sl-2cb-3ft
alphaF-160sl-2cb-3ft
alphaF-180sl-2cb-3ft
alphaF-30sl-3cb-3ft
alphaF-40sl-3cb-3ft
alphaF-50sl-3cb-3ft
alphaF-60sl-3cb-3ft
alphaF-70sl-3cb-3ft
alphaF-90sl-3cb-3ft
alphaF-110sl-3cb-3ft
alphaF-130sl-3cb-3ft
alphaF-150sl-3cb-3ft
alphaF-20sl-4cb-3ft
alphaF-30sl-4cb-3ft
alphaF-40sl-4cb-3ft
alphaF-50sl-4cb-3ft
alphaF-60sl-4cb-3ft
alphaF-80sl-4cb-3ft
alphaF-19sl-5cb-3ft
alphaF-29sl-5cb-3ft
alphaF-39sl-5cb-3ft
alphaF-15sl-6cb-3ft
alphaF-25sl-6cb-3ft
alphaF-35sl-6cb-3ft
alphaF-14sl-7cb-3ft
alphaF-24sl-7cb-3ft
""".split()


config = ExperimentConfig(base_path=os.path.join('..', 'logs'), experiment="mnist_idle0.1", seeds=["42", '78', '84'],
                          algorithms=["fedavg"], events=["global"],
                          lr_list=['5e-2', '1e-2'], alphas=["0.5"], n_clients_list=["7"],
                          availabilities=availabilities,
                          n_rounds="100", participations=["known"], biased_list=["2"], train_test="train")

results_df = get_exp_stats(config)



import ast

# Convert 'test_accuracy' column to lists if needed
results_df['test_accuracy'] = results_df['test_accuracy'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)



# # Loop over availability groups
# for availability_value, group in results_df.groupby("availability"):
    
#     plt.figure(figsize=(10, 6))
    
#     for _, row in group.iterrows():
#         label = f"lr={row['lr']} | seed={row['seed']}"
#         plt.plot(row['test_accuracy'], label=label)
    
#     plt.title(f"Test Accuracy Curves — availability = {availability_value}")
#     plt.xlabel("Epoch")
#     plt.ylabel("Test Accuracy")
#     plt.grid(True)
#     plt.legend()
#     plt.show()



In [15]:
results_df = results_df[['availability','final_test_accuracy', 'seed','lr']]
tmp = results_df.groupby(['availability']).final_test_accuracy.apply(np.vstack).to_frame().reset_index()
tmp['var_test_acc'] = tmp['final_test_accuracy'].apply(lambda x : x.var(axis=0))
tmp['max_test_acc'] = tmp['final_test_accuracy'].apply(lambda x: x.max(axis=0))
tmp = tmp[['availability', 'var_test_acc', 'max_test_acc']]
tmp["cb"] = tmp["availability"].str.extract(r"(\d+)cb").astype(int)
tmp["ft"] = tmp["availability"].str.extract(r"(\d+)ft").astype(int)
tmp["sl"] = tmp["availability"].str.extract(r"(\d+)sl").astype(int)
tmp

,availability,var_test_acc,max_test_acc,cb,ft,sl
0,alphaF-100sl-2cb-1ft,[0.17618077914091956],[99.08000230789185],2,1,100
1,alphaF-100sl-2cb-3ft,[0.16961402592367372],[98.97000193595886],2,3,100
2,alphaF-110sl-1cb-1ft,[0.10124695202229361],[99.04999732971191],1,1,110
3,alphaF-110sl-1cb-3ft,[0.12342212907097878],[99.14000034332275],1,3,110
4,alphaF-110sl-3cb-1ft,[0.25099011115297604],[98.47000241279602],3,1,110
...,...,...,...,...,...,...
74,alphaF-80sl-4cb-3ft,[17.546585376604856],[94.63000297546387],4,3,80
75,alphaF-90sl-1cb-1ft,[0.19024770680214295],[99.02999997138977],1,1,90
76,alphaF-90sl-1cb-3ft,[0.1720129349536137],[99.11999702453613],1,3,90
77,alphaF-90sl-3cb-1ft,[0.315899300512691],[98.68999719619751],3,1,90


In [17]:
cb_to_time = [40, 30, 20, 10, 5, 4, 3]

tmp = tmp[tmp["ft"] == 1]
tmp["T"] = tmp["cb"].apply(lambda x : cb_to_time[x-1])
tmp["sl"] = tmp["sl"] - tmp["T"]
tmp

/tmp/ipykernel_213645/3972810734.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tmp["T"] = tmp["cb"].apply(lambda x : cb_to_time[x-1])
/tmp/ipykernel_213645/3972810734.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tmp["sl"] = tmp["sl"] - tmp["T"]


,availability,var_test_acc,max_test_acc,cb,ft,sl,T
0,alphaF-100sl-2cb-1ft,[0.17618077914091956],[99.08000230789185],2,1,40,30
2,alphaF-110sl-1cb-1ft,[0.10124695202229361],[99.04999732971191],1,1,30,40
4,alphaF-110sl-3cb-1ft,[0.25099011115297604],[98.47000241279602],3,1,70,20
6,alphaF-120sl-2cb-1ft,[0.22251418768289474],[98.79000186920166],2,1,60,30
8,alphaF-130sl-1cb-1ft,[0.13015494861928148],[98.91999959945679],1,1,50,40
10,alphaF-130sl-3cb-1ft,[25.31671485498303],[91.97999835014343],3,1,90,20
12,alphaF-13sl-7cb-1ft,[17.102566691973053],[96.35000228881836],7,1,7,3
13,alphaF-140sl-2cb-1ft,[0.30869125709721673],[98.5700011253357],2,1,80,30
15,alphaF-14sl-6cb-1ft,[6.728355688973132],[97.79000282287598],6,1,6,4
16,alphaF-150sl-1cb-1ft,[0.12445845159210951],[98.86000156402588],1,1,70,40


In [26]:
import pandas as pd
df = tmp
# --- 1) Ensure list-like numeric columns are scalars (e.g., [74.87] -> 74.87) ---
for col in ["var_test_acc", "max_test_acc"]:
    df[col] = df[col].apply(lambda x: x[0] if isinstance(x, (list, tuple)) and len(x) > 0 else x)

# Make sure cb, sl, T are integers (optional but recommended)
df["cb"] = df["cb"].astype(int)
df["sl"] = df["sl"].astype(int)
df["T"]  = df["T"].astype(int)

# --- 2) Pivot to wide: rows are (cb, T), columns are sl, values are max_test_acc ---
wide = (df.pivot_table(index=["cb", "T"], columns="sl", values="max_test_acc", aggfunc="first")
          .sort_index()
          .sort_index(axis=1))

# Optional: nicer LaTeX column header like "sl=80" instead of just "80"
wide.columns = [f"sl={c}" for c in wide.columns]

# --- 3) Export to LaTeX ---
latex_str = wide.to_latex(
    index=True,
    na_rep="",
    float_format="%.2f",
    caption="Max test accuracy by cb, T and sl",
    label="tab:cb_T_sl",
    column_format="ll" + "r"*wide.shape[1]  # 2 left cols (cb,T) + right-aligned numeric cols
)

print(latex_str)


\begin{table}
\caption{Max test accuracy by cb, T and sl}
\label{tab:cb_T_sl}
\begin{tabular}{llrrrrrrrrrrrrrrrrrrrrrrrrrrrr}
\toprule
 &  & sl=21 & sl=30 & sl=41 & sl=50 & sl=55 & sl=60 & sl=61 & sl=65 & sl=70 & sl=75 & sl=80 & sl=81 & sl=85 & sl=90 & sl=95 & sl=100 & sl=101 & sl=105 & sl=110 & sl=115 & sl=120 & sl=121 & sl=125 & sl=130 & sl=135 & sl=140 & sl=145 & sl=150 \\
cb & T &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
1 & 40 &  &  &  &  &  &  &  &  & [74.94999766] &  &  &  &  & [74.87000227] &  &  &  &  & [75.52000284] &  &  &  &  & [75.44000149] &  &  &  & [75.55000186] \\
\cline{1-30}
2 & 30 &  &  &  &  &  &  &  & [74.83000159] &  &  &  &  & [74.91000295] &  &  &  &  & [75.01999736] &  &  &  &  & [74.98999834] &  &  &  & [75.29000044] &  \\
\cline{1-30}
3 & 20 &  &  &  &  &  & [74.29999709] &  &  &  &  & [74.87000227] &  &  &  &  & [74.58000183] &  &  &  &  & [74.81999993] &  &  &  &  & [75.2399981] &  &  \\
\cline{1-30}
4 &